# Med42-8B LoRA Fine-Tuning — Colab (free tier)

Runs the `lora_finetune/` pipeline from `kavyasridhar1501/hemorrhoids-rag-chatbot` without cloning the whole repo — it pulls only the files each task needs via the GitHub API.

Two fine-tuning tasks are available, both QLoRA on `m42-health/Llama3-Med42-8B`:

- **Task A — red-flag/triage JSON extraction (recommended, run this one).** Scoped classification task: given a patient message, output `{"red_flags": [...], "urgency": "...", "reasoning": "..."}`. Scored deterministically (precision/recall/F1, JSON-validity rate) against a Claude-labeled, hand-reviewable eval set — no LLM-judge noise. No RAG/vectorstore needed.
- **Task B — free-text answer imitation (appendix, optional).** The original task: distills the Claude+RAG chatbot's free-text answers into training data. Kept for reference — its one completed run was inconclusive (too few optimizer steps to move an 8B model, see `lora_finetune/README.md`), which is why Task A exists.

**Before running:**
1. `Runtime → Change runtime type → T4 GPU`
2. Open the key icon (🔑) in the left sidebar → add these Colab secrets, and toggle "Notebook access" on for each:
   - `GITHUB_TOKEN` — a GitHub personal access token with read access to this repo (Settings → Developer settings → Fine-grained tokens → scope it to just this repo, `Contents: Read-only`)
   - `HF_TOKEN` — Hugging Face token; also accept the license on https://huggingface.co/m42-health/Llama3-Med42-8B
   - `ANTHROPIC_API_KEY` — used to generate/label training data (Task A), and as the chatbot being distilled from + the judge in evaluation (Task B)
3. Task A needs nothing else. Task B additionally needs source medical documents or an existing `faiss_index/` to build the RAG vectorstore — see its appendix section.

## 1. Confirm GPU

In [1]:
!nvidia-smi

Sun Jul 19 00:28:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Secrets → environment variables

In [2]:
import os
from google.colab import userdata

os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 3. Install dependencies

Colab already ships a CUDA-matched `torch` — don't reinstall it, just add the rest.

In [3]:
!pip install -q transformers>=4.40.0 peft>=0.10.0 accelerate>=0.28.0 bitsandbytes>=0.43.0 datasets>=2.18.0 sentencepiece>=0.1.99
!pip install -q python-dotenv anthropic langchain langchain-anthropic langchain-huggingface sentence-transformers langchain-community langchain-text-splitters langchain-core faiss-cpu beautifulsoup4 requests pypdf markdown

## 4. Fetch only the files this pipeline needs

Pulls via the GitHub Contents API (works for private repos with a token) instead of a full `git clone`. Points at the `claude/fine-tuning-setup-review-yesk21` branch, where the extraction task lives — switch `GITHUB_BRANCH` to `main` once that branch is merged.

In [4]:
import os
import requests
from pathlib import Path

GITHUB_OWNER = "kavyasridhar1501"
GITHUB_REPO = "hemorrhoids-rag-chatbot"
GITHUB_BRANCH = "claude/fine-tuning-setup-review-yesk21"  # switch to "main" once merged
WORKDIR = Path("/content/hemorrhoids-rag-chatbot")

FILES_NEEDED = [
    "requirements.txt",
    "lora_finetune/requirements.txt",
    "patient_chatbot.py",
    "conversation_memory.py",
    "rag_setup.py",
    "test_runner.py",
    "testing_framework.py",
    "test_data/forum_test_cases.json",
    "test_data/google_test_cases.json",
    "test_data/manual_test_cases.json",
    "lora_finetune/config.py",
    "lora_finetune/extraction_schema.py",
    "lora_finetune/generate_extraction_questions.py",
    "lora_finetune/prepare_extraction_dataset.py",
    "lora_finetune/evaluate_extraction.py",
    "lora_finetune/prepare_dataset.py",
    "lora_finetune/train_lora.py",
    "lora_finetune/generate_lora.py",
    "lora_finetune/evaluate_lora.py",
    "lora_finetune/merge_adapter.py",
    "lora_finetune/generate_questions.py",
    "lora_finetune/run_pipeline.py",
    "medical_scraper.py",
]

def fetch_github_file(path: str):
    url = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/contents/{path}"
    headers = {
        "Authorization": f"Bearer {os.environ['GITHUB_TOKEN']}",
        "Accept": "application/vnd.github.v3.raw",
    }
    resp = requests.get(url, headers=headers, params={"ref": GITHUB_BRANCH})
    resp.raise_for_status()
    dest = WORKDIR / path
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(resp.content)
    print(f"✓ {path}")

WORKDIR.mkdir(parents=True, exist_ok=True)
for path in FILES_NEEDED:
    fetch_github_file(path)

✓ requirements.txt
✓ lora_finetune/requirements.txt
✓ patient_chatbot.py
✓ conversation_memory.py
✓ rag_setup.py
✓ test_runner.py
✓ testing_framework.py
✓ test_data/forum_test_cases.json
✓ test_data/google_test_cases.json
✓ test_data/manual_test_cases.json
✓ lora_finetune/config.py
✓ lora_finetune/extraction_schema.py
✓ lora_finetune/generate_extraction_questions.py
✓ lora_finetune/prepare_extraction_dataset.py
✓ lora_finetune/evaluate_extraction.py
✓ lora_finetune/prepare_dataset.py
✓ lora_finetune/train_lora.py
✓ lora_finetune/generate_lora.py
✓ lora_finetune/evaluate_lora.py
✓ lora_finetune/merge_adapter.py
✓ lora_finetune/generate_questions.py
✓ lora_finetune/run_pipeline.py
✓ medical_scraper.py


In [5]:
%cd /content/hemorrhoids-rag-chatbot

/content/hemorrhoids-rag-chatbot


---
## 5. Task A — Generate class-balanced red-flag messages

Generates ~40 messages per red flag, 80 routine (no red flags), and multi-flag combinations via Claude — class balance matters here, otherwise the model just learns to always predict "routine". Writes `test_data/extraction_messages.json`.

In [6]:
!python lora_finetune/generate_extraction_questions.py

Generating 40 messages for 'severe_pain'...
  +40
Generating 40 messages for 'heavy_bleeding'...
  +40
Generating 40 messages for 'fever'...
  +40
Generating 40 messages for 'prolonged_constipation'...
  +40
Generating 40 messages for 'black_stool'...
  +40
Generating 40 messages for 'dizziness'...
  +40
Generating 80 routine messages...
  +81
Generating 10 multi-flag messages for 'prolonged_constipation+black_stool'...
  +10
Generating 10 multi-flag messages for 'heavy_bleeding+fever'...
  +10
Generating 10 multi-flag messages for 'fever+prolonged_constipation'...
  +10
Generating 10 multi-flag messages for 'black_stool+dizziness'...
  +10
Generating 10 multi-flag messages for 'severe_pain+heavy_bleeding'...
  +10
Generating 10 multi-flag messages for 'dizziness+severe_pain'...
  +10

Wrote 381 messages to test_data/extraction_messages.json


## 6. Task A — Label + build train/val data

Labels each message with a **separate, blind** Claude call (it never sees which flag a message was generated for) and builds a stratified train/val split. Writes `lora_finetune/data/extraction_{train,val}.jsonl` plus a human-readable `extraction_val_for_review.jsonl`.

In [7]:
!python lora_finetune/prepare_extraction_dataset.py

  labeled 25/381
  labeled 50/381
  labeled 75/381
  labeled 100/381
  labeled 125/381
  labeled 150/381
  labeled 175/381
  labeled 200/381
  labeled 225/381
  labeled 250/381
  labeled 275/381
  labeled 300/381
  labeled 325/381
  labeled 350/381
  labeled 375/381

Wrote 313 train / 68 val examples to lora_finetune/data/
Review lora_finetune/data/extraction_val_for_review.jsonl by hand before trusting evaluate_extraction.py's numbers.


In [8]:
import json
for name in ["extraction_train", "extraction_val"]:
    path = f"lora_finetune/data/{name}.jsonl"
    with open(path) as f:
        n = sum(1 for _ in f)
    print(f"{name}: {n} examples")

extraction_train: 313 examples
extraction_val: 68 examples


**Before trusting any eval numbers below, skim this cell's output.** A Claude-labeled eval set is only as trustworthy as the labeler — this is the actual eval data in a readable format, not the tokenized training rows.

In [9]:
import json
with open("lora_finetune/data/extraction_val_for_review.jsonl") as f:
    rows = [json.loads(line) for line in f]
print(f"{len(rows)} eval examples\n")
for row in rows[:15]:
    print(f"[{row['target_category']}] {row['text'][:90]}")
    print(f"  -> {row['label']}\n")

68 eval examples

[dizziness] lightheaded and shaky since my last bowel movement, there was noticeably more blood this t
  -> {'red_flags': ['dizziness', 'heavy_bleeding'], 'urgency': 'emergency', 'reasoning': 'Increased bleeding with lightheadedness signals possible significant blood loss.'}

[prolonged_constipation] Haven't pooped in over 3 days now, feeling really bloated and my stomach hurts.
  -> {'red_flags': ['prolonged_constipation'], 'urgency': 'see_doctor_soon', 'reasoning': 'No bowel movement for over 3 days needs medical attention.'}

[dizziness] getting these dizzy spells randomly throughout the day, also constipated and hemorrhoids a
  -> {'red_flags': ['dizziness'], 'urgency': 'emergency', 'reasoning': 'Random dizziness spells are a critical symptom needing evaluation.'}

[black_stool] stool was dark black today, almost looks burnt. still have the constipation issue going on
  -> {'red_flags': ['black_stool'], 'urgency': 'emergency', 'reasoning': 'Black stool suggests po

## 7. Task A — Fine-tune

Uses `ExtractionLoRAConfig` in `lora_finetune/config.py` — shorter `max_seq_length` (completions are compact JSON, not paragraphs) and more epochs than the free-text task, since each step is cheaper. If you hit an OOM, lower `per_device_train_batch_size` to 1 (raise `gradient_accumulation_steps` to compensate) in that file, then re-run this cell.

In [10]:
!python lora_finetune/train_lora.py --task extraction

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 23.9k/23.9k [00:00<00:00, 43.8MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/6.08G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/11.1G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/16.1G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   2% 268M/16.1G [00:07<06:35, 39.9MB/s, 12.0MB/s  ]
Reconstructing (incomplete total...):   9% 1.48G/16.1G [00:22<06:17, 38.6MB/s, 26.4MB/s  ]
Reconstructing (incomplete total...):  11% 1.74G/16.1G [00:22<04:14, 56.3MB/s, 13.0MB/s  ]
Reconstructing (incomplete total...):  13% 2.08G/16.1G [00:39<03:50, 60.6MB/s, 36.8MB/s  ]
Reconstructing (incomplete total...):  21% 3.31G/16.1G [01:04<04:50, 43.8MB/s, 25.9MB/s  ]
Reconstructing (incomplete total...):  33% 5.33G/16.1G [01:29<01:41

## 8. Task A — Evaluate base vs. LoRA

Deterministic scoring against the (hand-reviewed, per step 6) eval set: strict JSON-validity rate, micro-averaged flag precision/recall/F1, per-flag breakdown, urgency exact-match accuracy, and strict exact-match rate. Loads the two 8B models one after another to fit in 16GB.

In [11]:
!python lora_finetune/evaluate_extraction.py

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   1% 2/291 [00:09<23:47,  4.94s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 291/291 [01:15<00:00,  3.85it/s]

Generating base Med42-8B responses for 68 examples...
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenizatio

In [12]:
import json
with open("test_results/lora_vs_base_extraction_results.json") as f:
    results = json.load(f)
print(json.dumps(results["comparison"], indent=2))

{
  "base_med42": {
    "n": 68,
    "strict_json_validity_rate": 0.926,
    "micro_precision": 0.849,
    "micro_recall": 0.89,
    "micro_f1": 0.869,
    "urgency_accuracy": 0.956,
    "exact_match_rate": 0.765,
    "per_flag": {
      "severe_pain": {
        "precision": 0.682,
        "recall": 1.0,
        "f1": 0.811,
        "support": 15
      },
      "heavy_bleeding": {
        "precision": 0.929,
        "recall": 0.929,
        "f1": 0.929,
        "support": 14
      },
      "fever": {
        "precision": 1.0,
        "recall": 0.692,
        "f1": 0.818,
        "support": 13
      },
      "prolonged_constipation": {
        "precision": 0.8,
        "recall": 0.889,
        "f1": 0.842,
        "support": 18
      },
      "black_stool": {
        "precision": 1.0,
        "recall": 1.0,
        "f1": 1.0,
        "support": 9
      },
      "dizziness": {
        "precision": 0.917,
        "recall": 0.846,
        "f1": 0.88,
        "support": 13
      }
    }
  }

---
## Appendix (optional): original free-text answer imitation task

The earlier task — distills the Claude+RAG chatbot's full free-text answers into training data and fine-tunes Med42-8B to imitate them, scored by an LLM judge. Its one completed run (34 train / 4 val examples, ~9 total optimizer steps) came back statistically indistinguishable from the base model — see `lora_finetune/README.md` for why, and why Task A above exists as the better-scoped alternative. Run this section only if you specifically want that comparison too.

### A1. Generate synthetic training questions + build the vectorstore

In [13]:
!python lora_finetune/generate_questions.py

Generating 12 questions for 'symptom_identification'...
Traceback (most recent call last):
  File "/content/hemorrhoids-rag-chatbot/lora_finetune/generate_questions.py", line 106, in <module>
    main()
  File "/content/hemorrhoids-rag-chatbot/lora_finetune/generate_questions.py", line 76, in main
    texts = generate_category_questions(client, category, QUESTIONS_PER_CATEGORY)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/hemorrhoids-rag-chatbot/lora_finetune/generate_questions.py", line 53, in generate_category_questions
    message = client.messages.create(
              ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anthropic/_utils/_utils.py", line 294, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anthropic/resources/messages/messages.py", line 1050, in create
    return self._post(
           ^^^^^^^^^^^
  File "/usr/local/l

In [ ]:
!python medical_scraper.py

import shutil
from pathlib import Path

Path("documents").mkdir(exist_ok=True)
for txt_file in Path("medical_articles").glob("*.txt"):
    shutil.copy(txt_file, "documents")

n_docs = len(list(Path("documents").glob("*.txt")))
print(f"{n_docs} articles copied into documents/")
assert n_docs > 0, "Scraper collected 0 articles - check medical_articles/blocked_urls.txt"

!python rag_setup.py

Starting medical article scraping...

Searching 18 trusted medical sources
Using direct URLs + web search


Searching: Mayo Clinic
  Using 4 known URLs...

    https://www.mayoclinic.org/diseases-conditions/hemorrhoids/diagnosis-treatment/drc-20360280
      Fetching URL...
      Access denied (403) - website blocking automated access
      Bookmark for manual download: https://www.mayoclinic.org/diseases-conditions/hemorrhoids/diagnosis-treatment/drc-20360280

    https://www.mayoclinic.org/diseases-conditions/constipation/diagnosis-treatment/drc-20354259
      Fetching URL...
      Access denied (403) - website blocking automated access
      Bookmark for manual download: https://www.mayoclinic.org/diseases-conditions/constipation/diagnosis-treatment/drc-20354259

    https://www.mayoclinic.org/diseases-conditions/hemorrhoids/symptoms-causes/syc-20360268
      Fetching URL...
      Access denied (403) - website blocking automated access
      Bookmark for manual download: https://www.

### A2. Build the training data

Runs the existing Claude + RAG chatbot over the test-case questions and saves its answers as fine-tuning targets.

In [ ]:
!python lora_finetune/prepare_dataset.py

In [ ]:
import json
for name in ["train", "val"]:
    path = f"lora_finetune/data/{name}.jsonl"
    with open(path) as f:
        n = sum(1 for _ in f)
    print(f"{name}: {n} examples")

### A3. Fine-tune

Defaults in `lora_finetune/config.py` (`LoRAConfig`) are already T4-safe (`compute_dtype="float16"`, `gradient_checkpointing=True`). If you hit an OOM, lower `per_device_train_batch_size` to 1 and/or `max_seq_length` to 512-768, then re-run this cell.

In [ ]:
!python lora_finetune/train_lora.py

### A4. Evaluate base vs. LoRA Med42-8B

Uses the same Claude LLM-as-judge as `test_runner.py`.

In [ ]:
!python lora_finetune/evaluate_lora.py

In [ ]:
import json
with open("test_results/lora_vs_base_results.json") as f:
    results = json.load(f)
print(json.dumps(results["comparison"], indent=2))

---
## 9. Persist adapters + results before the runtime recycles

Colab free tier is ephemeral — download these now, or mount Drive earlier and point `output_dir` in `config.py` there before training. Only zips/downloads what actually exists, so this is safe to run whether you did Task A, the appendix, or both.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

for name, path in [
    ("lora_adapter_extraction", "lora_finetune/adapter_extraction"),
    ("lora_adapter_chat", "lora_finetune/adapter"),
    ("lora_results", "test_results"),
]:
    if not Path(path).exists():
        print(f"skip {path} (not found)")
        continue
    shutil.make_archive(name, "zip", path)
    files.download(f"{name}.zip")